In [38]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils import extract_text
import re
from random import choice

In [39]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [40]:
file = ROOT / Path("examenes_quimica/2026_quimica/2026 - quimica - ordinaria titular titular quínica.pdf")

In [41]:
with open("primeras_lineas.txt", "w") as f:
    for folder in sorted((ROOT / "examenes_quimica").iterdir()):
        if folder.is_dir():
            chosen_pdf = Path(choice(list(folder.iterdir())))
            f.write("\t" + " - ".join(chosen_pdf.parts[-2:]) + "\n")
            with pdfplumber.open(chosen_pdf) as pdf:
                for p in pdf.pages:
                    lines = "\n".join(p.extract_text().split("\n")[:5]) + "\n\n"
                    f.write(lines)

In [42]:
def left_neighbor(char, chars_by_x1):
    """Carácter que termina justo donde empieza `char` (a menos de 1,5 pt) y a su misma altura."""
    x = round(char["x0"])
    candidates = [other for key in (x - 1, x, x + 1) for other in chars_by_x1.get(key, [])
                  if other is not char
                  and abs(other["x1"] - char["x0"]) < 1.5
                  and other["top"] < char["bottom"] and char["top"] < other["bottom"]]
    return min(candidates, key=lambda other: abs(other["x1"] - char["x0"]), default=None)

def extract_page_text(page):
    """Texto de la página convirtiendo los sub/superíndices en caracteres Unicode."""
    # índice de los caracteres por su borde derecho (redondeado) para encontrar vecinos rápido
    chars_by_x1 = {}
    for char in page.chars:
        chars_by_x1.setdefault(round(char["x1"]), []).append(char)

    base_of = {}       # id(índice) -> carácter normal al que acompaña (la "H" de "H₂")
    characters = []
    for char in sorted(page.chars, key=lambda char: char["x0"]):   # de izquierda a derecha
        left = left_neighbor(char, chars_by_x1)
        # si el vecino ya es un índice (el "¹" de "10⁻¹⁴"), la base es la de ese vecino
        base = base_of.get(id(left), left)
        if base is not None and char["size"] < 0.9 * base["size"]:
            # superíndice si su base queda claramente por encima de la del carácter normal
            table = SUPERSCRIPT if char["bottom"] < base["bottom"] - 2 else SUBSCRIPT
            base_of[id(char)] = base
            # se le da la altura de su base para que extract_text lo ponga en la misma línea
            char = dict(char, text=char["text"].translate(table), top=base["top"], bottom=base["bottom"])
        characters.append(char)
    # extract_text junta en una línea los caracteres cuyo top difiere como máximo y_tolerance
    # y dentro de cada línea los ordena por x0
    return extract_text(characters, y_tolerance=4)

In [43]:
re.search(r"\d{4}", "examen 2025 3456").group(0)

'2025'

In [62]:
def extract_exercises_from_pdf(file: Path) -> str:
    match_year = re.search(r"\d{4}", file.stem)
    if match_year:
        year = int(match_year.group(0))

    exam_text = f"{file.stem}\n\n"

    if year <= 2019:
        start_string = "OPCIÓN"
    else:
        start_string = r"BLOQUE|B\d{1}\."

    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            page_text = extract_page_text(page)
            
            match = re.search(start_string, page_text)
            if match:
                page_text = page_text[match.start():]
                page_text = (page_text.replace("OPCIÓN A", "Opción A\n")
                             .replace("OPCIÓN B", "Opción B\n")
                             )

            exam_text += f"{page_text}\n\n"

    return exam_text

Diccionarios de sustitución de caracteres con `str.maketrans()`

In [45]:
SUBSCRIPT = str.maketrans("0123456789+-–‒−", "₀₁₂₃₄₅₆₇₈₉₊₋₋₋₋")
SUPERSCRIPT = str.maketrans("0123456789+-–‒−", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁻⁻⁻")

In [ ]:
file = ROOT / "examenes_quimica/2022_quimica/2017 - quimica - titular septiembre 6.pdf"

In [57]:
file_2022 = ROOT / "examenes_quimica/2022_quimica/2022 - quimica - impreso ord titular .pdf"

In [53]:
sept_2017_exercises = extract_exercises_from_pdf(file)

In [63]:
junio_2022_exercises = extract_exercises_from_pdf(file_2022)

In [54]:
print(sept_2017_exercises)

2017 - quimica - titular septiembre 6

Opción A

1.- Formule o nombre los siguientes compuestos: a) Dihidruro de cobalto; b) Hipoyodito de mercurio(II);
c) Ácido butanodioico; d) SrO₂; e) H₃AsO₄; f) CH₃CH₂CH₂NH₂.
2.- a) Justifique cuál de las siguientes especies, Li⁺ y He, tiene mayor radio.
b) Razone cuál de los siguientes elementos, O y N, tiene mayor afinidad electrónica.
c) Justifique cuál de los siguientes elementos, Na y Cl, tiene mayor energía de ionización.
3.- Razone la veracidad o falsedad de las siguientes afirmaciones:
a) Si a una disolución saturada de una sal insoluble se le añade uno de los iones que la forman, disminuye la
solubilidad.
b) Dos iones de cargas iguales y de signos opuestos forman un precipitado cuando el producto de sus
concentraciones es igual a su producto de solubilidad.
c) Para desplazar el equilibrio de solubilidad hacia la formación de más sólido insoluble, se extrae de la disolución
parte del precipitado.
4.- Justifique si las siguientes afirmacione

In [64]:
print(junio_2022_exercises)

2022 - quimica - impreso ord titular 

BLOQUE A (Formulación)
Puntuación máxima: 1,5 puntos
En este bloque se plantean 2 preguntas de las que debe responder SOLAMENTE 1.
La pregunta elegida tiene un valor máximo de 1,5 puntos.
A1. Formule o nombre los siguientes compuestos:
a) Bromato de aluminio; b) Sulfuro de antimonio(V); c) 1,1-Dicloro-2-metilciclohexano; d) PtO₂; e) Cr(OH)₃; f) CH₃NO₂
A2. Formule o nombre los siguientes compuestos:
a) Nitrato de hierro(III); b) Hidróxido de estaño(IV); c) Tricloroetanamida; d) CaCl₂; e) HClO₃; f) CH₃CH₂CH₂COOCH₂CH₃
BLOQUE B (Cuestiones)
Puntuación máxima: 4,5 puntos
En este bloque se plantean 6 cuestiones de las que debe responder SOLAMENTE 3.
Cada cuestión, a su vez, consta de tres apartados.
Cada cuestión tendrá un valor máximo de 1,5 puntos (0,5 puntos por apartado).
B1. Conteste las siguientes cuestiones relativas a un átomo con Z= 17 y A= 35.
a) Indique el número de protones, neutrones y electrones.
b) Escriba su configuración electrónica e i

In [ ]:
# Primera línea de la cabecera que se repite en cada página (termina en la línea que contiene "CURSO")
HEADER_START = re.compile(r"UNIVERSIDADES DE ANDALUC|PRUEBA DE (ACCESO|EVALUACI)")
# "OPCIÓN A" (2010-2019) o "BLOQUE A" (2020-2024)
SECTION = re.compile(r"(?:OPCI[ÓO]N|BLOQUE) ([A-Z])\b")
# Inicio de pregunta: "1.-" (2010-2019), "PREGUNTA 1.-" (2025-) o "A1." (2020-2024)
QUESTION = re.compile(r"(?:PREGUNTA )?(\d+)\s*\.\s*[-–]|([A-Z])(\d+)\.\s")


def split_questions_from_file(path):
    """Devuelve una lista de preguntas: dicts con "section", "number" y "text"."""
    with pdfplumber.open(path) as pdf:
        text = "\n".join(extract_page_text(page) for page in pdf.pages)

    questions = []
    section = None     # letra de la OPCIÓN/BLOQUE actual (None si el examen no tiene)
    current = None     # pregunta a la que se añaden las líneas (None = descartar líneas)
    in_header = False
    for line in text.splitlines():
        if HEADER_START.match(line):           # empieza la cabecera: se salta...
            in_header = True
        elif in_header:                        # ...hasta la línea "CURSO 20XX-20XX" incluida
            in_header = "CURSO" not in line
        elif line.startswith("Instruccion"):   # instrucciones: se descartan hasta la siguiente pregunta
            current = None
        elif match := SECTION.search(line):    # nueva opción/bloque
            section, current = match[1], None
        elif match := QUESTION.match(line):    # nueva pregunta
            current = {"section": section or match[2],
                       "number": int(match[1] or match[3]),
                       "text": line}
            questions.append(current)
        elif current:                          # continuación de la pregunta actual
            current["text"] += "\n" + line
    return questions

In [26]:
questions = split_questions_from_file(file)

In [28]:
questions

[]

In [27]:
len(questions)

0

In [17]:
for q in questions:
    print(f"[{q['section'] or ''}{q['number']}]", q["text"], sep="\n", end="\n" + "-" * 10 + "\n\n")